# Functions: partial application and currying

Consider this simple method doing calculations on its arguments:

In [1]:
def calculateSimple(x:Double,y:Double,z:Double) = x*4+y/z

defined function calculateSimple

In [19]:
calculateSimple(3,4,5)

res18: Double = 12.8

We can modify it to allow multiple argument lists. In principle, it looks very much the same:

In [2]:
def calculate(x:Double)(y:Double)(z:Double) = x*4+y/z

defined function calculate

In [2]:
calculate(3,4,5)

-- [E007] Type Mismatch Error: cmd2.sc:1:21 ------------------------------------
1 |val res2 = calculate(3,4,5)
  |                     ^^^^^
  |                     Found:    (Int, Int, Int)
  |                     Required: Double
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

However now the call works differently, as we need to apply it to three different arguments:

In [3]:
calculate(3)(4)(5)

res2: Double = 12.8

## One interesting feature: custom control structures

A consequence of having parameter groups is that we can create our own control structures.

Consider an `ìf` where the condition may fail:

In [19]:
val num = "30,3"

if ( num.toInt > 20) {
    println(s"the value is $i")
}

: 

We can create a new condition function, `ifSafe` whose second group parameter is a call-by-name parameter, i.e. a block of code.

In [14]:
import scala.util.Try

def ifSafe[A](condition: => Boolean)(codeBlock: => A) =
  val cond = Try(condition)
  if (cond.isSuccess && condition) then 
    codeBlock
    true
  else false

import scala.util.Try


defined function ifSafe

We can now use our new `ifSafe` structure:

In [20]:
val i="20"

ifSafe ( i.toInt > 20) {
    println(s"the value is $i")
}

i: Int = 20
res19_1: Boolean = false

### Partial application

We will see now another consequnce: **partial application**. Let's consider again the ``calculate` function:

In [22]:
def calculate(x:Double)(y:Double)(z:Double) = x*4+y/z

defined function calculate

Interestingly, now we can perform a partial application of this function, for instance only providing the first argument.

In [4]:
val partial = calculate(3)

partial: Double => Double => Double = ammonite.$sess.cmd3$Helper$$Lambda$2459/239495609@33db669f

As a result we have another function that we can call afterwards with the remaining parameter lists.

In [5]:
partial(4)(5)

res4: Double = 12.8

We can rewrite the function to make it more explicit that it will actually return another function:

In [23]:
def calculate2(x:Double) =  (y:Double) => (z:Double) =>  x*4+y/z

defined function calculate2

In [24]:
calculate2(3)(4)(5)

res23: Double = 12.8

Also works with partial applicaiton of two arguments:

In [25]:
val partial2 = calculate2(3)(4)
partial2(5)
partial2(6)

partial2: Double => Double = ammonite.$sess.cmd22$$Lambda$2515/1811175596@ad50cfe
res24_1: Double = 12.8
res24_2: Double = 12.666666666666666

## Currying

This works based on existing results from the **theory of currying**.

What this theory says is that a function that takes multiple arguments can be translated into a series of function calls that each take a single argument. 

In pseudocode, this means that an expression like this:

```
result = f(x)(y)(z)
```

is mathematically the same as something like this:

```
f1 = f(x)
f2 = f1(y)
result = f2(z)
```

We can define curried functions in this manner:

In [26]:
 def plus(a: Int)(b: Int) = a + b

defined function plus

And we can then "load" the function with one value:

In [27]:
val plus5 = plus(5)

plus5: Int => Int = ammonite.$sess.cmd26$Helper$$Lambda$2526/1390665748@6ff19df9

And we can reuse this as a new function:

In [28]:
plus5(6)

res27: Int = 11

In [41]:
plus5(9)

res40: Int = 14

In [42]:
plus(_)

res41: Int => Int => Int = ammonite.$sess.cmd41$Helper$$Lambda$2591/602927151@2ef855f3

Another example with string formatting of HTML:

In [43]:
def wrap(prefix:String)(html:String)(suffix:String) = 
  prefix + html + suffix

defined function wrap

In [39]:
 val hello = "Hi there!!"
 val result = wrap("<div>")(hello)("</div>")

hello: String = "Hi there!!"
result: String = "<div>Hi there!!</div>"

In [40]:
val wrapDiv = wrap("<div>")(_:String)("</div>")

wrapDiv: String => String = ammonite.$sess.cmd39$Helper$$Lambda$2576/1461767610@31d0c1c1

In [34]:
wrapDiv("Bye all!!")

res33: String = "<div>Bye all!!</div>"

In [37]:
wrap("<body>")
    (wrapDiv("Hey"))
    ("</body>")

res36: String = "<body><div>Hey</div></body>"

In [45]:
val square = scala.math.pow(_,2)

square: Double => Double = ammonite.$sess.cmd44$Helper$$Lambda$2597/590552960@2c0efa5

In [46]:
square(4)

res45: Double = 16.0

In [47]:
val cube = scala.math.pow(_,3)

cube: Double => Double = ammonite.$sess.cmd46$Helper$$Lambda$2600/1979077596@2b92aeb5

In [48]:
cube(5)

res47: Double = 125.0

### Curried functions

When we decompose function arguments in that way we are "currying" the functions. This has been long studied in lambda calculus, but we can reuse this concept in functional programming languages. 

There are actually some convenient ways of transforming a normal function into a curried one:

In [1]:
def calculateSimple(x:Double,y:Double,z:Double) = x*4+y/z

defined function calculateSimple

In [9]:
val calcCurried = calculateSimple.curried

calcCurried: Double => Double => Double => Double = scala.Function3$$Lambda$2479/516933160@53d4f3c4

Now the partial invocation are possible. This is convenient for situations where existing methods are invoked with fixed parameters, or partially reused. 

In [10]:
calcCurried(3)

res9: Double => Double => Double = scala.Function3$$Lambda$2481/1970429272@350ff2a9

In [11]:
val calcWith3 = calcCurried(3)
calcWith3(4)(5)

calcWith3: Double => Double => Double = scala.Function3$$Lambda$2481/1970429272@1fbde9e1
res10_1: Double = 12.8

The opposite is also possible, we ca "uncurry" functions as follows:

In [26]:
val unCurriedCalc = Function.uncurried(partial)

unCurriedCalc: (Double, Double) => Double = scala.Function$$$Lambda$2599/1509295930@75e7beb4

In [27]:
unCurriedCalc(4,5)

res26: Double = 12.8

### Tupling functions

In certain occasions it becomes interesting to see function arguments as a tuple. For instance this grading function:

In [13]:
def finalGrade(g1:Int,g2:Int,g3:Int) = 
  1+((g1+g2+g3)/3)/20

defined function finalGrade

In [14]:
finalGrade(70,80,30)

res13: Int = 4

If the grades are given as a Tuple, it becomes cumbresome to make the call by accessing each individual element.

In [15]:
val grades =(70,80,30)

finalGrade(grades._1,grades._2,grades._3)

grades: (Int, Int, Int) = (70, 80, 30)
res14_1: Int = 4

Instead there is a way to convert the method signature as receiving a Tuple:

In [13]:
def finalGrade(g1:Int,g2:Int,g3:Int) = 
  1+((g1+g2+g3)/3)/20

defined function finalGrade

In [35]:
val finalGradeT = Function.tupled(finalGrade)

finalGradeT: ((Int, Int, Int)) => Int = scala.Function$$$Lambda$2631/1734451390@5f919c56

In [36]:
finalGradeT(grades)

res35: Int = 4

In [37]:
finalGrade.tupled(grades)

res36: Int = 4